In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from classifier_weather.src.dataset import get_transforms, TestDataset, build_idx_to_target
from classifier_weather.src.train import train, val
from classifier_weather.src.predict import save_predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
TRAIN_DATA_DIR = '../data/train/train'
TEST_DATA_DIR = '../data/test/test'

In [ ]:
# создание датасета и разделение на обучение и валидацию
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(TRAIN_DATA_DIR, transform=train_transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

In [ ]:
# подсчет среднего и стандартного отклонения для трэйна
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=False)

mean = torch.zeros(3)
std = torch.zeros(3)
n_pixels = 0

for images, _ in train_loader:
    b, c, h, w = images.shape

    n_pixels += b * h * w

    mean += images.sum(dim=[0, 2, 3])
    std += (images ** 2).sum(dim=[0, 2, 3])

mean /= n_pixels
std = torch.sqrt(std / n_pixels - mean ** 2)

print("Mean:", mean)
print("Std:", std)

In [ ]:
print('Train_size: ', len(train_dataset))
print('Val_size:', len(val_dataset))

In [ ]:
data_transforms = get_transforms(mean, std)

train_indices = train_dataset.indices # индексы трэйн датасета
val_indices = val_dataset.indices # индексы вал датасета

train_dataset = torch.utils.data.Subset(datasets.ImageFolder(TRAIN_DATA_DIR, transform=data_transforms['train']), train_indices)
val_dataset = torch.utils.data.Subset(datasets.ImageFolder(TRAIN_DATA_DIR, transform=data_transforms['val']), val_indices)

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, persistent_workers=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, persistent_workers=True)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, X):
        identity = self.shortcut(X)

        out = self.conv1(X)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity

        out = self.relu(out)

        return out

In [ ]:
class My_ResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.first = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 64)
        )

        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128)
        )

        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(256, 3)

    def forward(self, X):
        x = self.first(X)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

In [ ]:
model = My_ResNet()

In [ ]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=3e-4)

In [ ]:
def plot_training_curves(train_losses, train_scores, val_scores=None, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(train_losses, label='Train Loss')
    axes[0].set_title('Loss по эпохам')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(train_scores, label='Train F1')
    if val_scores is not None:
        axes[1].plot(val_scores, label='Val F1')
    axes[1].set_title('F1 по эпохам')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 score')
    axes[1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Сохранено: {save_path}")
    plt.show()

In [ ]:
model, train_losses, train_scores, val_scores = train(
    model, train_dataloader, val_dataloader, optimizer, criterion, num_epoch=25
)

plot_training_curves(
    train_losses, train_scores, val_scores,
    save_path='../results/training_curves_my_resnet.png'
)

In [ ]:
test_dataset = TestDataset(TEST_DATA_DIR, transform=data_transforms['val'])

test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

In [ ]:
class_to_target = {
    'rain': 0,
    'fog': 1,
    'snow': 2
}

model_idx_to_target = {
    dataset.class_to_idx[k]: v for k, v in class_to_target.items()
}

model.eval()

filenames = []
predictions = []

with torch.no_grad():
    for X_batch, names in tqdm(test_dataloader):
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = outputs.argmax(dim=1)

        filenames.extend(names)
        predictions.extend([model_idx_to_target[p.item()] for p in preds])

save_predict(dataset, predictions, filenames, output_path='../results/my_resnet_pred.csv')